# TraceCir — P0 (TAPR baseline E0)

Notebook chạy trên Google Colab (GPU). Thiết kế để **chạy lại an toàn nhiều lần**:
- Dữ liệu (COCO zip) chỉ tải **1 lần**, lưu trên Drive, các lần sau tự bỏ qua nếu đã có.
- Cache đặc trưng (`build_feature_cache`) tự **resume** nếu bị ngắt giữa chừng.
- Nếu Colab mất kết nối/reset máy ảo: cứ **Run All** lại từ đầu, các bước đã xong sẽ tự bỏ qua hoặc tiếp tục đúng chỗ dở dang.

## Bước 0 — Mount Drive + kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!!! Chua bat GPU: vao Runtime > Change runtime type > GPU roi chay lai tu dau !!!')

## Bước 1 — Lấy code (clone lần đầu / pull nếu đã có)

In [ ]:
import os

REPO_URL = 'https://github.com/khaidz123321/TraceCir.git'
REPO_DIR = '/content/TraceCir'

if os.path.isdir(REPO_DIR):
    print('Repo da co san, chay git pull lay ban moi nhat...')
    !git -C {REPO_DIR} pull
else:
    print('Clone repo lan dau...')
    !git clone {REPO_URL} {REPO_DIR}

print()
print('!!! Neu vua git pull ma co cap nhat code p0/ hoac models/, hay Restart session')
print('    (Runtime > Restart session) roi chay lai tu Buoc 0, vi Python cache module da import.')

In [ ]:
%cd /content/TraceCir
!pip install -q -r requirements.txt

## Bước 2 — Dữ liệu CIRCO

- Ảnh COCO `unlabeled2017.zip` (~19.5GB): tải **1 lần duy nhất** vào Drive, các lần sau tự bỏ qua.
- Giải nén vào ổ cục bộ Colab (`/content/data/...`) mỗi phiên — vì ổ cục bộ bị xóa khi máy ảo reset, còn file `.zip` gốc trên Drive thì không mất.

In [ ]:
import os

DRIVE_DATA_DIR = '/content/drive/MyDrive/TraceCir_data'
COCO_ZIP_PATH = f'{DRIVE_DATA_DIR}/unlabeled2017.zip'

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

if os.path.exists(COCO_ZIP_PATH):
    size_gb = os.path.getsize(COCO_ZIP_PATH) / 1e9
    print(f'unlabeled2017.zip da co san tren Drive ({size_gb:.1f} GB) -- bo qua tai lai.')
else:
    print('Chua co file tren Drive, dang tai ve (~19.5GB, chi lam 1 lan)...')
    !wget -q --show-progress http://images.cocodataset.org/zips/unlabeled2017.zip -O {COCO_ZIP_PATH}
    print('Da tai xong va luu vao Drive.')

In [ ]:
import os

LOCAL_CIRCO_DIR = '/content/data/CIRCO'
IMG_DIR = f'{LOCAL_CIRCO_DIR}/COCO2017_unlabeled/unlabeled2017'

if os.path.isdir(IMG_DIR) and len(os.listdir(IMG_DIR)) == 123403:
    print('Anh da giai nen du 123403 file roi, bo qua giai nen lai.')
else:
    print('Dang giai nen anh vao o cuc bo Colab (vai phut)...')
    os.makedirs(f'{LOCAL_CIRCO_DIR}/COCO2017_unlabeled', exist_ok=True)
    !unzip -q {COCO_ZIP_PATH} -d {LOCAL_CIRCO_DIR}/COCO2017_unlabeled/
    n = len(os.listdir(IMG_DIR))
    print(f'Da giai nen {n} anh (ky vong 123403).')

In [ ]:
import os

if os.path.isdir(f'{LOCAL_CIRCO_DIR}/annotations'):
    print('Annotation CIRCO da co san, bo qua.')
else:
    print('Dang lay annotation CIRCO...')
    if not os.path.isdir('/content/CIRCO_annotations_repo'):
        !git clone -q https://github.com/miccunifi/CIRCO.git /content/CIRCO_annotations_repo
    !cp -r /content/CIRCO_annotations_repo/annotations {LOCAL_CIRCO_DIR}/annotations

print()
print(os.listdir(LOCAL_CIRCO_DIR))
print(os.listdir(f'{LOCAL_CIRCO_DIR}/annotations'))

## Bước 3 — Cache đặc trưng OpenCLIP (bước nặng nhất, ~1-2 giờ)

Lưu thẳng vào Drive. Nếu bị ngắt giữa chừng (crash, mất kết nối), **chạy lại đúng cell này** sẽ tự động tiếp tục từ chỗ dở dang, không làm lại từ đầu.

In [ ]:
import sys
sys.path.insert(0, '/content/TraceCir')

from src.models.openclip_utils import load_openclip
from src.data.datasets import CIRCODataset
from src.p0.feature_cache import build_feature_cache, FeatureCache

device = 'cuda'
model, preprocess, tokenizer = load_openclip(device=device)

ds_classic = CIRCODataset(LOCAL_CIRCO_DIR, 'val', 'classic', preprocess)
print('So anh trong index:', len(ds_classic))

FEATURE_CACHE_DIR = f'{DRIVE_DATA_DIR}/features/circo'

build_feature_cache(
    model, ds_classic,
    output_dir=FEATURE_CACHE_DIR,
    id_key='image_id',
    device=device,
    batch_size=64,
    num_workers=2,
)

In [ ]:
# Kiem tra nhanh cache da xong dung chua
cache = FeatureCache(FEATURE_CACHE_DIR)
print('So anh trong cache:', len(cache))
print('global shape:', cache.global_features.shape, cache.global_features.dtype)
print('local shape:', cache.local_features.shape, cache.local_features.dtype)
assert len(cache) == 123403, 'Cache chua du anh, kiem tra lai Buoc 3'
print('OK - cache day du.')

## Bước 4 — Transition Compiler (Qwen2.5-VL-7B-Instruct)

Cần chạy trên ~220 câu truy vấn CIRCO validation. **Sau khi chạy xong, bắt buộc phải tự audit thủ công** (mục 5.3 giao thức P0) trước khi tin tưởng dùng cho Bước 5.

In [ ]:
from src.p0.compiler import load_compiler, run_compiler_batch

compiler_model, compiler_processor = load_compiler(
    model_name='Qwen/Qwen2.5-VL-7B-Instruct',
    device='cuda',
    load_in_4bit=True,
)

In [ ]:
from src.data.datasets import CIRCODataset
from PIL import Image

# Dung dataset CIRCO 'relative' nhung KHONG qua preprocess (compiler can anh PIL goc)
ds_query = CIRCODataset(LOCAL_CIRCO_DIR, 'val', 'relative', preprocess=lambda x: x)
print('So cau truy van CIRCO val:', len(ds_query))

queries = []
for item in ds_query:
    query_id = str(item['reference_img_id'])
    queries.append((item['reference_image'], item['relative_caption'], query_id))

COMPILED_PATH = f'{DRIVE_DATA_DIR}/circo_compiled.jsonl'
records = run_compiler_batch(compiler_model, compiler_processor, queries, COMPILED_PATH)
print(f'Da chay compiler cho {len(records)} cau, luu tai {COMPILED_PATH}')

### ⚠️ Audit thủ công (bắt buộc, không tự động hóa được)

Mở file `circo_compiled.jsonl` (trên Drive), đọc ngẫu nhiên ít nhất 100 dòng, tự đánh giá `target`/`atoms` có đúng với `modification` không. Gán nhãn Correct/Partially/Incorrect. Nếu tỉ lệ Correct < ~85%, quay lại sửa `COMPILER_PROMPT` trong `src/p0/compiler.py`, `git push`, rồi chạy lại Bước 4.

In [ ]:
# Cong cu nho de xem nhanh vai ket qua compiler ngay trong notebook (khong thay the audit day du)
import json

with open(COMPILED_PATH, 'r', encoding='utf-8') as f:
    lines = [json.loads(l) for l in f]

for r in lines[:10]:
    print('Modification:', r['modification'])
    print('Target:', r['target'])
    print('Atoms:', r['atoms'])
    print('parse_ok:', r['parse_ok'])
    print('---')

## Bước 5 — Chạy E0, ra Bảng 4 (mAP@5/@10/@25/@50)

**Chỉ chạy bước này sau khi audit ở Bước 4 đạt ≥85% Correct.**

In [ ]:
!python -m src.p0.run_e0 \
    --dataset circo --split val --data-root {LOCAL_CIRCO_DIR} \
    --feature-cache-dir {FEATURE_CACHE_DIR} \
    --compiled-queries-path {COMPILED_PATH}